#### LSTM 자연어 모델 문제 
1. data 폴더 안에 ratings_train.txt 파일을 로드 
2. 텍스트 정규화 함수를 이용하여 document 컬럼의 텍스트들을 정규화 
    - 특수문자 제거, 2칸 이상의 공백을 1칸의 공백으로 대체, 문자 좌우의 공백을 제거 
3. 공백 테스트를 제거 
4. 결측치 제거 
5. 중복된 데이터 제거 
6. 상위 5000개의 데이터를 추출 
7. komoran을 이용해서 데이터 토큰화 
    - 품사 NNP NNG VV VA MAG SL 만 사용
8. 단어 사전을 생성한다. (최소 출현 횟수는 2회)
8. Dataset을 생성(인코딩 작업 결합)하고 collate_fn을 생성하여 패딩토큰을 채워준다. 
9. DataLoader를 생성 
10. 8:2의 비율로 train, vali데이터셋으로 나눠준다. 
10. LSTM 학습 모델을 생성 
    - Embedding() -> LSTM() -> Linea()
    - LSTM에서는 마지막 히든층을 이용하여 선형 모델에 대입
11. 에폭의 횟수는 50회로 모델을 검증

In [34]:
import pandas as pd 
import torch 
import torch.nn as nn
import torch.optim as optim 
from torch.utils.data import Dataset, DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence
from konlpy.tag import Komoran
from collections import Counter
from tqdm import tqdm
import re 

In [35]:
df = pd.read_csv("../data/ratings_train.txt", sep='\t')

In [36]:
# 텍스트 정규화 함수 
def normalize(text):
    text = re.sub(r'[^가-힣0-9a-zA-Z\s\.]', ' ', str(text))
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [37]:
df['document'] = df['document'].map(normalize)

In [38]:
df = df.loc[ ~(df['document'] == ''),  ]

In [39]:
df.info()

<class 'pandas.DataFrame'>
Index: 149564 entries, 0 to 149999
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype
---  ------    --------------   -----
 0   id        149564 non-null  int64
 1   document  149564 non-null  str  
 2   label     149564 non-null  int64
dtypes: int64(2), str(1)
memory usage: 4.6 MB


In [40]:
df.dropna(inplace=True)
df.drop_duplicates('document', inplace=True)

In [41]:
komoran = Komoran()

def tokenize(text):
    allow_pos = ['NNP', 'NNG', 'VV', 'VA', 'MAG', 'SL']

    result = []
    for word, pos in komoran.pos(text):
        if pos in allow_pos:
            result.append(word)
    return result

In [42]:
df = df[:5000]

In [43]:
tokenized_setences = [ tokenize(text) for text in df['document'] ]

In [44]:
# 단어 사전 구축 
vocab = {
    '<PAD>' : 0, 
    '<UNK>' : 1
}
all_tokens = [ token for tokens in tokenized_setences for token in tokens ]
token_count = Counter(all_tokens)

for token, count in token_count.items():
    if count >= 2:
        vocab[token] = len(vocab)
vocab

{'<PAD>': 0,
 '<UNK>': 1,
 '더빙': 2,
 '진짜': 3,
 '짜증': 4,
 '나': 5,
 '목소리': 6,
 '포스터': 7,
 '초딩': 8,
 '영화': 9,
 '오버': 10,
 '연기': 11,
 '가볍': 12,
 '이야기': 13,
 '솔직히': 14,
 '재미': 15,
 '없': 16,
 '평점': 17,
 '조정': 18,
 '돋보이': 19,
 '스파이더맨': 20,
 '늙': 21,
 '보이': 22,
 '하': 23,
 '너무나': 24,
 '막': 25,
 '떼': 26,
 '초등학교': 27,
 '학년': 28,
 '용': 29,
 '별': 30,
 '반개': 31,
 '아깝': 32,
 '원작': 33,
 '긴장감': 34,
 '제대로': 35,
 '살리': 36,
 '욕': 37,
 '나오': 38,
 '생활': 39,
 '이': 40,
 '정말': 41,
 '발로': 42,
 '납치': 43,
 '반복': 44,
 '드라마': 45,
 '가족': 46,
 '못하': 47,
 '사람': 48,
 '모이': 49,
 '액션': 50,
 '있': 51,
 '안': 52,
 '왜': 53,
 '낮': 54,
 '꽤': 55,
 '보': 56,
 '헐리우드': 57,
 '너무': 58,
 '볼': 59,
 '때': 60,
 '눈물': 61,
 '나서': 62,
 '죽': 63,
 '향수': 64,
 '자극': 65,
 '감성': 66,
 '절제': 67,
 '멜로': 68,
 '달인': 69,
 '이다': 70,
 '울': 71,
 '건너': 72,
 '이범수': 73,
 '드럽': 74,
 '좋': 75,
 '기사': 76,
 '로만': 77,
 '보다': 78,
 '자꾸': 79,
 '잊어버리': 80,
 '취향': 81,
 '존중': 82,
 '극장': 83,
 '가장': 84,
 '노': 85,
 '재': 86,
 '감동': 87,
 '스토리': 88,
 '어거지': 89,
 '매번': 90,
 '긴장'

In [45]:
# Dataset, collate_fn, DataLoader 생성 
class LSTMDataset(Dataset):
    def __init__(self, tokenized_texts, labels, vocab):
        self.labels = labels.values
        self.data = [
            [vocab.get(token, vocab['<UNK>']) for token in tokens] 
            for tokens in tokenized_texts
        ]
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        return torch.tensor(self.data[idx], dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.long)

In [46]:
dataset = LSTMDataset(tokenized_setences, df['label'], vocab)
train_size = int(len(dataset) * 0.8)
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])


In [47]:
def collate_fn(batch):
    # text_list = [item[0] for item in batch]
    # labels = [item[1] for item in batch]
    text_list, labels = zip(*batch)
    padded_texts = pad_sequence(text_list, batch_first=True, padding_value=vocab['<PAD>'])
    labels = torch.tensor(labels, dtype=torch.long)
    return padded_texts, labels

In [48]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle = True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle = True, collate_fn= collate_fn)

In [49]:
list(train_loader)

[(tensor([[2031,   22, 1597,  ...,    0,    0,    0],
          [  23,  312,   16,  ...,    0,    0,    0],
          [  17,  764, 1065,  ...,    0,    0,    0],
          ...,
          [1360,  603,    1,  ...,    0,    0,    0],
          [1894, 1804,    1,  ...,    0,    0,    0],
          [ 272,  683,    9,  ...,    0,    0,    0]]),
  tensor([0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1,
          1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 1,
          0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0])),
 (tensor([[ 911,   16,    9,  ...,    0,    0,    0],
          [  58,  271,  272,  ...,    0,    0,    0],
          [ 312,   52, 3164,  ...,    0,    0,    0],
          ...,
          [ 162,   52,   56,  ...,    0,    0,    0],
          [ 680, 1217,   56,  ...,    5,    0,    0],
          [ 868,  578,    1,  ...,    0,    0,    0]]),
  tensor([0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0,
          0

In [63]:
class LSTMCLF(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_size, num_classes, dropout = 0.5, head_type = 'last'):
        super().__init__()

        self.head_type = head_type

        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=vocab['<PAD>'])

        # 자비에르 초기화 
        torch.nn.init.xavier_uniform_(self.emb.weight)

        self.lstm = nn.LSTM(emb_dim, hidden_size, batch_first=True)
        # 과적합 방지용 dropout
        self.dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(hidden_size, num_classes)
    
    def forward(self, x):
        embedding = self.emb(x)

        # LSTM 결과 값 A, (B,C)
        lstm_out, (hidden, cell) = self.lstm(embedding)
        if self.head_type == 'last':
            last_hidden = hidden.squeeze(0)
        elif self.head_type == 'mean':
            # 모든 층의 값들의 평균을 구한다. 
            # lstm_out -> [batch_size, seq_len, hidden_size]
            last_hidden = torch.mean( lstm_out, dim=1 )    # [batch_size, hidden_size]
        elif self.head_type == 'max':
            last_hidden, _ = torch.max(lstm_out, dim = 1)

        dropout_hidden = self.dropout(last_hidden)

        # return self.fc(last_hidden)
        return self.fc(dropout_hidden)

In [64]:
model = LSTMCLF(len(vocab), emb_dim=64, hidden_size=128, num_classes=2, head_type='max')
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = 0.001)

In [65]:
epochs = 50

for epoch in range(epochs):
    model.train()
    train_loss = 0
    correct_train = 0 
    total_train = 0 

    for inputs, labels in tqdm(train_loader, desc = f"Epoch {epoch+1} / {epochs} "):
        optimizer.zero_grad()
        output = model(inputs)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        pred = torch.argmax(output, dim=1)
        correct_train += (pred == labels).sum().item()
        total_train += labels.size(0)
    train_acc = (correct_train / total_train) * 100
    avg_train_loss  = train_loss / len(train_loader)

    model.eval()
    val_loss = 0
    correct_val = 0
    total_val = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            output = model(inputs)
            loss = criterion(output, labels)

            val_loss += loss.item()
            pred = torch.argmax(output, dim = 1)
            correct_val += (pred == labels).sum().item()
            total_val += labels.size(0)

    val_acc = (correct_val / total_val) * 100
    avg_val_loss = val_loss / len(val_loader)

    if (epoch + 1) % 10 == 0:
        print(f"LSTM 에폭 결과 : Train Loss {round(avg_train_loss, 4)} Train Acc {train_acc} " )
        print(f"LSTM 에폭 결과 : Vali Loss {round(avg_val_loss, 4)} Vali Acc {val_acc}")

Epoch 10 / 50 : 100%|██████████| 63/63 [00:00<00:00, 70.72it/s]


LSTM 에폭 결과 : Train Loss 0.148 Train Acc 95.05 
LSTM 에폭 결과 : Vali Loss 0.7713 Vali Acc 71.1


Epoch 20 / 50 : 100%|██████████| 63/63 [00:00<00:00, 79.13it/s]


LSTM 에폭 결과 : Train Loss 0.1246 Train Acc 95.975 
LSTM 에폭 결과 : Vali Loss 1.1043 Vali Acc 71.89999999999999


Epoch 30 / 50 : 100%|██████████| 63/63 [00:00<00:00, 73.76it/s]


LSTM 에폭 결과 : Train Loss 0.1018 Train Acc 96.65 
LSTM 에폭 결과 : Vali Loss 1.1229 Vali Acc 69.0


Epoch 40 / 50 : 100%|██████████| 63/63 [00:00<00:00, 81.94it/s]


LSTM 에폭 결과 : Train Loss 0.1052 Train Acc 96.0 
LSTM 에폭 결과 : Vali Loss 1.4076 Vali Acc 69.69999999999999


Epoch 50 / 50 : 100%|██████████| 63/63 [00:00<00:00, 71.19it/s]


LSTM 에폭 결과 : Train Loss 0.1062 Train Acc 96.075 
LSTM 에폭 결과 : Vali Loss 1.5846 Vali Acc 68.89999999999999
